# 03. Silver Layer Processing

## Banking Transaction Lakehouse

**Purpose:** Validate Bronze transaction records and prepare clean, deduplicated data for downstream analytics.

### Scope
- Apply transaction validation rules.
- Identify the latest transaction versions.
- Separate valid and invalid records.
- Preserve invalid records in Quarantine with reasons.
- Reconcile Bronze, Silver and Quarantine record counts.
- Merge valid records into Silver.

**Implementation:** Silver processing logic is maintained in `src/silver/silver_processor.py`.

In [0]:
from pyspark.sql import functions as F

from src.silver.silver_processor import process_silver

BRONZE_TABLE = "workspace.default.dev_bronze_transactions"
SILVER_TABLE = "workspace.default.dev_silver_transactions"
QUARANTINE_TABLE = "workspace.default.dev_quarantine_transactions"

batch_id = "bronze_demo_20261106_01"

print("Silver processing notebook initialized.")

In [0]:
# Create dedicated tables using the existing development schemas
spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.default.demo_silver_transactions
    LIKE workspace.default.dev_silver_transactions
""")

spark.sql("""
    CREATE TABLE IF NOT EXISTS workspace.default.demo_quarantine_transactions
    LIKE workspace.default.dev_quarantine_transactions
""")

SILVER_TABLE = "workspace.default.demo_silver_transactions"
QUARANTINE_TABLE = "workspace.default.demo_quarantine_transactions"

# Read the Bronze records for this batch
bronze_current_batch = (
    spark.table(BRONZE_TABLE)
    .filter(F.col("batch_id") == batch_id)
)

print("Bronze records ready for Silver:", bronze_current_batch.count())

In [0]:
# Process Bronze records through the Silver layer
silver_result = process_silver(
    spark=spark,
    bronze_current_batch=bronze_current_batch,
    silver_table=SILVER_TABLE,
    quarantine_table=QUARANTINE_TABLE
)

# Capture processing results
silver_valid_batch = silver_result["silver_valid_batch"]
quarantine_batch = silver_result["quarantine_batch"]

print("Bronze records:", silver_result["bronze_count"])
print("Valid records:", silver_result["valid_count"])
print("Quarantined records:", silver_result["quarantine_count"])
print("Duplicate records:", silver_result["duplicate_count"])

In [0]:
# Inspect quarantined transactions and their rejection reasons
display(
    quarantine_batch
    .select(
        "transaction_id",
        "customer_id",
        "amount",
        "transaction_type",
        "dq_reason"
    )
    .orderBy("transaction_id")
)

In [0]:
# Verify persisted Silver records
silver_verification = (
    spark.table(SILVER_TABLE)
    .filter(F.col("batch_id") == batch_id)
)

# Verify persisted Quarantine records
quarantine_verification = (
    spark.table(QUARANTINE_TABLE)
    .filter(F.col("batch_id") == batch_id)
)

print("Persisted Silver records:", silver_verification.count())
print("Persisted Quarantine records:", quarantine_verification.count())

print("Silver records:")
display(silver_verification)

print("Quarantine records:")
display(quarantine_verification)